# 01 — Data Cleaning

Bản chỉnh theo **FINAL**: giữ thông tin hoàn trả để tính giá trị ròng, loại trùng, loại mã phí/điều chỉnh, ghi log số dòng trước–sau. Không dùng thông tin xảy ra sau ngày dự báo để sửa feature lịch sử.

In [1]:
import pandas as pd
import numpy as np

input_path = '../data/raw/online_retail_II.csv'
output_path = '../data/interim/transactions_cleaned.csv'

df = pd.read_csv(input_path)
print('Số dòng dữ liệu thô:', len(df))
print('Khoảng thời gian:', df['InvoiceDate'].min(), '->', df['InvoiceDate'].max())
print(df.head())

Số dòng dữ liệu thô: 1067371
Khoảng thời gian: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00
  Invoice StockCode  ... Customer ID         Country
0  489434     85048  ...     13085.0  United Kingdom
1  489434    79323P  ...     13085.0  United Kingdom
2  489434    79323W  ...     13085.0  United Kingdom
3  489434     22041  ...     13085.0  United Kingdom
4  489434     21232  ...     13085.0  United Kingdom

[5 rows x 8 columns]


In [2]:
# Chuẩn hóa kiểu dữ liệu
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Invoice'] = df['Invoice'].astype(str).str.strip()
df['StockCode'] = df['StockCode'].astype(str).str.strip()

cleaning_log = []
def log_step(step, before, after):
    cleaning_log.append({'step': step, 'before': before, 'after': after, 'removed': before-after})
    print(f'{step}: {before:,} -> {after:,} (loại {before-after:,})')

# 1) Loại dòng không thể định danh khách hàng / ngày giao dịch
before = len(df)
df_clean = df.dropna(subset=['Customer ID', 'InvoiceDate']).copy()
log_step('Thiếu Customer ID/InvoiceDate', before, len(df_clean))

# 2) Loại dòng trùng hoàn toàn
before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
log_step('Dòng trùng hoàn toàn', before, len(df_clean))

# 3) Loại giá bằng 0; giữ Quantity âm vì đó là thông tin hoàn trả
before = len(df_clean)
df_clean = df_clean[df_clean['Price'] > 0].copy()
log_step('Price <= 0', before, len(df_clean))

# 4) Loại các mã phí/điều chỉnh không phải hàng hóa
# Các mã này không dùng để tính UniqueProducts/TotalItems của sản phẩm.
non_product_codes = {
    'POST', 'DOT', 'M', 'D', 'S', 'BANK CHARGES',
    'ADJUST', 'AMAZONFEE', 'CRUK', 'B', 'GIFT'
}
before = len(df_clean)
df_clean = df_clean[~df_clean['StockCode'].str.upper().isin({x.upper() for x in non_product_codes})].copy()
log_step('Mã phí/điều chỉnh', before, len(df_clean))

# 5) Phân biệt mua và hoàn trả/hủy. KHÔNG xóa return.
df_clean['is_return'] = (
    df_clean['Invoice'].str.upper().str.startswith('C') |
    (df_clean['Quantity'] < 0)
)
df_clean['transaction_type'] = np.where(df_clean['is_return'], 'return', 'purchase')

# Revenue có dấu: mua dương, hoàn trả âm.
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

print('\nPhân loại giao dịch:')
print(df_clean['transaction_type'].value_counts())
print('Doanh thu ròng trong dữ liệu cleaned:', round(df_clean['Revenue'].sum(), 2))
print('\nLOG LÀM SẠCH:')
print(pd.DataFrame(cleaning_log))

Thiếu Customer ID/InvoiceDate: 1,067,371 -> 824,364 (loại 243,007)


Dòng trùng hoàn toàn: 824,364 -> 797,885 (loại 26,479)
Price <= 0: 797,885 -> 797,815 (loại 70)


Mã phí/điều chỉnh: 797,815 -> 794,454 (loại 3,361)

Phân loại giao dịch:
transaction_type
purchase    776857
return       17597
Name: count, dtype: int64
Doanh thu ròng trong dữ liệu cleaned: 16371798.26

LOG LÀM SẠCH:
                            step   before   after  removed
0  Thiếu Customer ID/InvoiceDate  1067371  824364   243007
1           Dòng trùng hoàn toàn   824364  797885    26479
2                     Price <= 0   797885  797815       70
3              Mã phí/điều chỉnh   797815  794454     3361


In [3]:
# Quy tắc dùng xuyên suốt pipeline:
# - Frequency/Recency: dựa trên hóa đơn MUA hợp lệ (purchase).
# - Monetary: doanh thu RÒNG = mua + hoàn trả đã biết trước thời điểm cắt dữ liệu.
# - TotalItems: số lượng RÒNG = Quantity có dấu trong khoảng lịch sử.
# - Return sau ngày dự báo KHÔNG được dùng để sửa feature trước ngày dự báo.

df_clean = df_clean.sort_values('InvoiceDate').reset_index(drop=True)
df_clean.to_csv(output_path, index=False)
print(f'\nĐã lưu: {output_path}')
print('Số dòng cuối:', len(df_clean))
print(df_clean.head())


Đã lưu: ../data/interim/transactions_cleaned.csv
Số dòng cuối: 794454
  Invoice StockCode  ... transaction_type  Revenue
0  489434     85048  ...         purchase     83.4
1  489434    79323P  ...         purchase     81.0
2  489434    79323W  ...         purchase     81.0
3  489434     22041  ...         purchase    100.8
4  489434     21232  ...         purchase     30.0

[5 rows x 11 columns]
